# Extracting spatiotemporal metrics for the Blob in FOSI and Blob-analogs

## Imports

In [1]:
# Imports
import xarray as xr
import numpy as np
import pandas as pd
import json
import pickle
import os
import matplotlib.pyplot as plt
from scipy import stats
import importlib
import analysis_functions as afuncs
import cesm2_lens_utils

importlib.reload(afuncs)
importlib.reload(cesm2_lens_utils)

## Functions

In [ ]:
def calc_frac_overlap(one_obj, mask):
    one_obj_binary = xr.where(one_obj > 0, 1., 0)
    overlap = one_obj_binary + mask
    overlap_only = xr.where(overlap > 1, 1., 0)
    overlap_frac = overlap_only.sum(dim=('lat','lon')) / mask.sum()
    if overlap_frac.size > 0 and overlap_frac.notnull().any():
        return overlap_frac.max().item()
    return 0.0

In [ ]:
def process_one_member_mld(member_id, labels_full, analog_ids_for_member, all_analog_mld, NEPac_mask):
    if not analog_ids_for_member:
        print(f"Member {member_id}: no analogs, skipping")
        return all_analog_mld

    member_labels = labels_full.isel(ensemble_member=member_id)

    ds_hist, ds_fut = afuncs.get_ds_var('/glade/campaign/cgd/cesm/CESM2-LE/ocn/proc/tseries/month_1/HMXL/', 'HMXL', 'ocn', member_id)

    hmxl_hist_cropped = ds_hist.HMXL.isel(nlat=lat_slice, nlon=lon_slice)
    hmxl_fut_cropped = ds_fut.HMXL.isel(nlat=lat_slice, nlon=lon_slice)
    hmxl_combined = xr.concat([hmxl_hist_cropped, hmxl_fut_cropped], dim='time').sel(time=slice('1979-01-01','2020-12-01'))
    hmxl_combined = hmxl_combined.rename({'nlat': 'nlat_t', 'nlon': 'nlon_t'}).assign_coords(
        nlat_t=NEPac_mask.nlat_t, nlon_t=NEPac_mask.nlon_t
    ) / 100.0

    for analog_id in analog_ids_for_member:
        key = (member_id, analog_id)
        if key in all_analog_mld:
            continue

        times_present = member_labels.time.where((member_labels == analog_id).any(dim=('lat','lon')), drop=True)
        if len(times_present) == 0:
            continue

        hmxl_masked = (hmxl_combined.sel(time=times_present) * NEPac_mask).compute()
        hmxl_masked = hmxl_masked.where(hmxl_masked != 0)

        all_analog_mld[key] = {
            'mld_mean': float(hmxl_masked.mean()),
            'mld_max': float(hmxl_masked.max()),
            'mld_p90': float(hmxl_masked.quantile(0.9)),  # NEW
        }

    print(f"Member {member_id}: done ({len(analog_ids_for_member)} analogs)")
    return all_analog_mld

In [ ]:
def compute_analog_characteristics(labels_da, analog_id, ssta_da, mld_da, cell_area_da):
    mask = (labels_da == analog_id)
    times_present = labels_da.time.where(mask.any(dim=('lat','lon')), drop=True)
    if len(times_present) == 0:
        return None

    mask_trimmed = mask.sel(time=times_present)
    ssta_masked = ssta_da.sel(time=times_present).where(mask_trimmed)
    mld_masked = mld_da.sel(time=times_present).where(mask_trimmed) if mld_da is not None else None
    extent_ts = (mask_trimmed.astype(int) * cell_area_da).sum(dim=('lat','lon'))

    result = {
        'duration': len(times_present),
        'intensity_mean': float(ssta_masked.mean()),
        'intensity_max': float(ssta_masked.max()),
        'extent_mean': float(extent_ts.mean()),
        'extent_max': float(extent_ts.max()),
        'extent_cumulative': float(extent_ts.sum()),
    }
    if mld_masked is not None:
        result['mld_mean'] = float(mld_masked.mean())
        result['mld_max'] = float(mld_masked.max())
    return result

## FOSI THE BLOB AND BLOB 2.0

In [8]:
# Fixed footprint (mask_3)
mask_3 = xr.open_dataset('mean_mask_3.nc').mhw_obj
mask_3_binary = xr.where(mask_3 > 0.1, 1, 0)

In [9]:
fosi_blobs = xr.open_dataset('fosi_blobs_r2.nc').labels  # already time-restricted? check below
fosi_blobs_check = fosi_blobs.sel(time=slice('1979-01','2020-12'))  # match your analysis period

unique_ids = np.unique(fosi_blobs_check.max(dim=('lat','lon')).compute().data)
unique_ids = unique_ids[~np.isnan(unique_ids)]
unique_ids = unique_ids[unique_ids > 0]

print(f"Total detected NEP MHW objects in FOSI (1979-2020): {len(unique_ids)}")

fosi_analog_overlaps = []
for object_id in unique_ids:
    one_obj = fosi_blobs_check.where(fosi_blobs_check == object_id, drop=False)
    overlap_val = calc_frac_overlap(one_obj, mask_3_binary)
    fosi_analog_overlaps.append((object_id, overlap_val))

fosi_analogs_50pct = [oid for oid, ov in fosi_analog_overlaps if ov > 0.5]
print(f"FOSI objects exceeding 50% canonical footprint overlap: {len(fosi_analogs_50pct)}")
print(f"Object IDs: {fosi_analogs_50pct}")

Total detected NEP MHW objects in FOSI (1979-2020): 106
FOSI objects exceeding 50% canonical footprint overlap: 2
Object IDs: [94.0, 110.0]


In [10]:
# Characterize object 110 
# when did it occur, how long, how intense?
object_id_2 = 110.0
times_present_2 = fosi_blobs_check.time.where((fosi_blobs_check == object_id_2).any(dim=('lat','lon')), drop=True)

print(f"Object 110 duration: {len(times_present_2)} months")
print(f"Start: {str(times_present_2.values[0])[:10]}")
print(f"End: {str(times_present_2.values[-1])[:10]}")

Object 110 duration: 10 months
Start: 2019-05-15
End: 2020-02-15


In [15]:
mask_2 = (fosi_blobs_check == object_id_2)
mask_2_trimmed = mask_2.sel(time=times_present_2)

regridded_ssta_no_trend_NEP_da = regridded_ssta_no_trend_NEP['__xarray_dataarray_variable__']

ssta_masked_2 = regridded_ssta_no_trend_NEP_da.sel(time=times_present_2).where(mask_2_trimmed)
extent_ts_2 = (mask_2_trimmed.astype(int) * cell_area_da_fosi).sum(dim=('lat','lon'))

print(f"Intensity mean: {float(ssta_masked_2.mean()):.2f}°C")
print(f"Intensity max: {float(ssta_masked_2.max()):.2f}°C")
print(f"Extent mean: {float(extent_ts_2.mean()):,.0f} km²")
print(f"Extent max: {float(extent_ts_2.max()):,.0f} km²")

Intensity mean: 0.94°C
Intensity max: 2.64°C
Extent mean: 12,043,308 km²
Extent max: 17,833,427 km²


In [16]:
# What fraction of the canonical footprint does it actually overlap?
for object_id in [94.0, 110.0]:
    one_obj = fosi_blobs_check.where(fosi_blobs_check == object_id, drop=False)
    overlap_val = calc_frac_overlap(one_obj, mask_3_binary)
    print(f"Object {object_id}: overlap = {overlap_val:.2f}")

Object 94.0: overlap = 0.61
Object 110.0: overlap = 0.69


In [17]:
mask_2 = (fosi_blobs_check == 110.0)
mask_2_trimmed = mask_2.sel(time=times_present_2)

ssta_masked_2 = regridded_ssta_no_trend_NEP_da.sel(time=times_present_2).where(mask_2_trimmed)
extent_ts_2 = (mask_2_trimmed.astype(int) * cell_area_da_fosi).sum(dim=('lat','lon'))

print(f"Intensity mean: {float(ssta_masked_2.mean()):.2f}°C")
print(f"Intensity max: {float(ssta_masked_2.max()):.2f}°C")
print(f"Extent mean: {float(extent_ts_2.mean()):,.0f} km²")
print(f"Extent max: {float(extent_ts_2.max()):,.0f} km²")

overlap_val_110 = calc_frac_overlap(fosi_blobs_check.where(fosi_blobs_check == 110.0, drop=False), mask_3_binary)
print(f"Canonical footprint overlap: {overlap_val_110:.2f}")

Intensity mean: 0.94°C
Intensity max: 2.64°C
Extent mean: 12,043,308 km²
Extent max: 17,833,427 km²
Canonical footprint overlap: 0.69


In [18]:
# Onset centroid - is it Gulf-of-Alaska concentrated, matching Blob 2.0's documented pattern?
first_time_110 = times_present_2.values[0]
mask_at_onset = mask_2.sel(time=first_time_110)

lat2d, lon2d = xr.broadcast(mask_at_onset.lat, mask_at_onset.lon)
w = mask_at_onset.where(mask_at_onset)
centroid_lat = float((lat2d*w).sum()/w.sum())
centroid_lon = float((lon2d*w).sum()/w.sum())
print(f"Onset centroid: {centroid_lat:.1f}°N, {centroid_lon:.1f}°E")

Onset centroid: 28.6°N, 211.3°E


In [20]:
lat_vals = mask_at_onset_110.where(mask_at_onset_110==1, drop=True).lat.values
print(f"Latitude range of masked region at onset: {lat_vals.min():.1f} to {lat_vals.max():.1f}")

# Rough check: fraction of masked area north of 45N (Gulf of Alaska) vs south of 25N (tropical/subtropical)
mask_gulf = mask_at_onset_110.where(mask_at_onset_110.lat > 45, drop=False).sum()
mask_tropical = mask_at_onset_110.where(mask_at_onset_110.lat < 25, drop=False).sum()
mask_total = mask_at_onset_110.sum()
print(f"Fraction of masked area north of 45°N (Gulf of Alaska): {float(mask_gulf/mask_total)*100:.1f}%")
print(f"Fraction of masked area south of 25°N (tropical/subtropical): {float(mask_tropical/mask_total)*100:.1f}%")

Latitude range of masked region at onset: 7.1 to 62.7
Fraction of masked area north of 45°N (Gulf of Alaska): 29.5%
Fraction of masked area south of 25°N (tropical/subtropical): 66.0%


## Load data

In [3]:
# Filepaths for the four detrending methods (CONFIRMED CORRECT this session)
base_paths = {
    'linear': '/glade/work/cmendiola/data_conv_lin_trend',
    'quadratic': '/glade/work/cmendiola/data_quad_trend',
    'ensmean': '/glade/work/cmendiola/data_ens_mean',
    'noseas': '/glade/work/cmendiola/data_rm_seasonalcycle_mean',
}

RADIUS_INDEX_FOR_2DEG = 1  # confirmed: index 1 = 2° structuring element, consistent across all four methods

# SSTa variable names per method (confirmed this session)
ssta_varnames = {
    'linear': '__xarray_dataarray_variable__',
    'quadratic': '__xarray_dataarray_variable__',
    'ensmean': 'SST',
    'noseas': '__xarray_dataarray_variable__',
}

In [4]:
# Load all four methods: labels + SSTa (radius-corrected, time-restricted to 1979-2020)
da_mhwobj_labels_by_method = {}
ssta_notrend_by_method = {}

for method, base_path in base_paths.items():
    mhwobj_paths = [f'{base_path}/ens_{i}_mhwobj.nc' for i in range(100)]
    ssta_paths = [f'{base_path}/ens_{i}_ssta.nc' for i in range(100)]

    da_mhwobj = xr.open_mfdataset(mhwobj_paths, combine='nested', concat_dim='ensemble_member')
    da_mhwobj_labels_by_method[method] = da_mhwobj.labels.sel(radius=RADIUS_INDEX_FOR_2DEG).sel(time=slice('1979-01','2020-12')).compute()

    ssta_notrend_by_method[method] = xr.open_mfdataset(ssta_paths, combine='nested', concat_dim='ensemble_member').compute()

print("All four methods loaded.")
print({m: da_mhwobj_labels_by_method[m].sizes for m in base_paths})

All four methods loaded.
{'linear': Frozen({'ensemble_member': 100, 'time': 504, 'lat': 64, 'lon': 81}), 'quadratic': Frozen({'ensemble_member': 100, 'time': 504, 'lat': 64, 'lon': 81}), 'ensmean': Frozen({'ensemble_member': 100, 'time': 504, 'lat': 64, 'lon': 81}), 'noseas': Frozen({'ensemble_member': 100, 'time': 504, 'lat': 64, 'lon': 81})}


In [5]:
# Canonical footprint (mask_3) - used throughout for Blob-analog identification and area-averaging
mask_3 = xr.open_dataset('mean_mask_3.nc').mhw_obj
mask_3_binary = xr.where(mask_3 > 0.1, 1, 0)

In [6]:
# Cell area weighting (CESM2-LENS grid)
R = 6371.0
grid_ref = da_mhwobj_labels_by_method['linear']
dlat = np.deg2rad(np.mean(np.diff(grid_ref.lat.values)))
dlon = np.deg2rad(np.mean(np.diff(grid_ref.lon.values)))
cell_area = (R**2) * dlat * dlon * np.cos(np.deg2rad(grid_ref.lat))
cell_area_da = xr.DataArray(cell_area.values, coords={'lat': grid_ref.lat}, dims='lat')

print(f"Equatorial gridcell area check: {float(cell_area_da.sel(lat=grid_ref.lat.values[len(grid_ref.lat)//2])):.1f} km²")

Equatorial gridcell area check: 11881.4 km²


In [7]:
# Blob-analog IDs (50% overlap threshold, index 4 of thresholds array) - CORRECTED for all four methods
thresholds = np.arange(0.1, 0.81, 0.1)

with open('object_id_ls_greenmask_linear_corrected.json', 'r') as f:
    linear_all_thresholds = json.load(f)
with open('object_id_ls_greenmask_ensmean_corrected.json', 'r') as f:
    ensmean_all_thresholds = json.load(f)
with open('object_id_ls_greenmask_quadratic_corrected.json', 'r') as f:
    quadratic_all_thresholds = json.load(f)
with open('object_id_ls_greenmask_noseas_corrected.json', 'r') as f:
    noseas_all_thresholds = json.load(f)

blob_analog_ids = {
    'linear': linear_all_thresholds[4],
    'ensmean': ensmean_all_thresholds[4],
    'quadratic': ensmean_all_thresholds[4],
    'noseas': ensmean_all_thresholds[4],
}

In [12]:
# FOSI Blob (Label 94, confirmed 20-month window: May 2014 - Dec 2015)
fosi_blobs = xr.open_dataset('fosi_blobs_r2.nc').labels
object_id = 94.0
blob_times = fosi_blobs.time.where((fosi_blobs == object_id).any(dim=('lat','lon')), drop=True)

regridded_ssta_no_trend_NEP = xr.open_dataset('/glade/derecho/scratch/cassiacai/regridded_ssta_no_trend_NEP.nc')

# Sanity check - should match your confirmed event max of 2.34°C
print(regridded_ssta_no_trend_NEP.sel(time='2015-07').max().values)
print(f"n_months in blob_times: {len(blob_times)}")  # should be 20

<bound method Mapping.values of <xarray.Dataset> Size: 12B
Dimensions:                        ()
Coordinates:
    z_t                            float32 4B ...
Data variables:
    __xarray_dataarray_variable__  float64 8B 2.345>
n_months in blob_times: 20


## Extract characteristics for EVERY Blob-analog (primary method: linear)

In [11]:
results_path = 'analog_mld_results.pkl'
if os.path.exists(results_path):
    with open(results_path, 'rb') as f:
        all_analog_mld = pickle.load(f)
    print(f"Resuming - {len(all_analog_mld)} analogs already processed")
else:
    all_analog_mld = {}

In [33]:
lat_slice = slice(204, 367)
lon_slice = slice(190, 280)

mean_image_pop_files = [f'/glade/derecho/scratch/cassiacai/mean_image_{i}_POP.nc' for i in range(1, 8)]
mean_images_pop = [xr.open_dataset(file) for file in mean_image_pop_files]
masks_pop = [xr.where(image.__xarray_dataarray_variable__ > 0.1, 1, 0) for image in mean_images_pop]
NEPac_MHW_renamed_latlon = masks_pop[2].isel(nlat=lat_slice, nlon=lon_slice).rename({'nlat': 'nlat_t', 'nlon': 'nlon_t'})

In [30]:
labels_full = da_mhwobj_labels_by_method['linear']

In [31]:
if os.path.exists(results_path):
    os.remove(results_path)
all_analog_mld = {}

In [28]:
member0_keys = [k for k in all_analog_mld.keys() if k[0] == 0]
print(f"Member 0 has {len(member0_keys)} analogs with MLD computed")

for k in member0_keys:
    print(k, all_analog_mld[k])

Member 0 has 4 analogs with MLD computed
(0, 28.0) {'mld_mean': 35.1125372057195, 'mld_max': 123.23858642578125}
(0, 43.0) {'mld_mean': 40.123765836332545, 'mld_max': 158.23272705078125}
(0, 58.0) {'mld_mean': 56.40374647746785, 'mld_max': 206.50076293945312}
(0, 85.0) {'mld_mean': 40.022922241983515, 'mld_max': 93.28509521484375}


In [34]:
for member_id in range(100):
    print(member_id)
    all_analog_mld = process_one_member_mld(member_id, labels_full, blob_analog_ids['linear'][member_id], all_analog_mld, NEPac_MHW_renamed_latlon)
    with open(results_path, 'wb') as f:
        pickle.dump(all_analog_mld, f)

print(f"Done. Total analogs with MLD: {len(all_analog_mld)}")

0
Member 0: done (4 analogs)
1
Member 1: done (2 analogs)
2
Member 2: done (4 analogs)
3
Member 3: done (3 analogs)
4
Member 4: done (5 analogs)
5
Member 5: done (5 analogs)
6
Member 6: done (2 analogs)
7
Member 7: done (3 analogs)
8
Member 8: done (3 analogs)
9
Member 9: done (3 analogs)
10
Member 10: done (4 analogs)
11
Member 11: done (2 analogs)
12
Member 12: done (4 analogs)
13
Member 13: done (2 analogs)
14
Member 14: done (3 analogs)
15
Member 15: done (4 analogs)
16
Member 16: done (3 analogs)
17
Member 17: done (3 analogs)
18
Member 18: done (3 analogs)
19
Member 19: done (2 analogs)
20
Member 20: done (4 analogs)
21
Member 21: done (4 analogs)
22
Member 22: done (3 analogs)
23
Member 23: done (4 analogs)
24
Member 24: done (3 analogs)
25
Member 25: done (3 analogs)
26
Member 26: done (4 analogs)
27
Member 27: done (4 analogs)
28
Member 28: done (6 analogs)
29
Member 29: done (4 analogs)
30
Member 30: done (3 analogs)
31
Member 31: done (2 analogs)
32
Member 32: done (3 analog

In [ ]:
# FOSI Blob MLD (raw): mean=57.8m, max=225.7m, p90=103.8m

In [1]:
# # Extract characteristics for EVERY Blob-analog (primary method: linear)
# all_analog_characteristics = {}
# labels_full = da_mhwobj_labels_by_method['linear']
# ssta_full = ssta_notrend_by_method['linear']['__xarray_dataarray_variable__']

# for member_id in range(100):
#     print(member_id)
#     member_labels = labels_full.isel(ensemble_member=member_id)
#     member_ssta = ssta_full.isel(ensemble_member=member_id)
#     for analog_id in blob_analog_ids['linear'][member_id]:
#         result = compute_analog_characteristics(member_labels, analog_id, member_ssta, None, cell_area_da)
#         if result:
#             all_analog_characteristics[(member_id, analog_id)] = result

# print(f"Total analogs characterized: {len(all_analog_characteristics)}")

In [113]:
save_bundle = {
    'all_analog_characteristics': all_analog_characteristics,   # duration, intensity, extent, MLD per analog
    'blob_fosi_characteristics': blob_fosi_characteristics,      # FOSI Blob's own values
    # 'analog_climate_stats': analog_climate_stats,                # Niño3.4/PDO per analog
    # 'blob_climate': blob_climate,                                # FOSI Blob's own Niño3.4/PDO values
}

with open('/glade/derecho/scratch/cassiacai/section_3_2_analog_data.pkl', 'wb') as f:
    pickle.dump(save_bundle, f)

print("Saved section_3_2_analog_data.pkl")
print(f"  - {len(all_analog_characteristics)} analogs with physical characteristics")

Saved section_3_2_analog_data.pkl
  - 331 analogs with physical characteristics


In [39]:
for key, mld_stats in all_analog_mld.items():
    if key in all_analog_characteristics:
        all_analog_characteristics[key].update(mld_stats)

print(f"Analogs with full characteristics (incl. MLD): "
      f"{sum(1 for v in all_analog_characteristics.values() if 'mld_mean' in v)} / {len(all_analog_characteristics)}")

Analogs with full characteristics (incl. MLD): 331 / 331


In [40]:
sample_keys = list(all_analog_characteristics.keys())[:5]
for k in sample_keys:
    print(k, all_analog_characteristics[k])

(0, 28.0) {'duration': 4, 'intensity_mean': 1.1449258083532783, 'intensity_max': 2.054659360954304, 'extent_mean': 9719089.75639243, 'extent_max': 16034480.422872044, 'extent_cumulative': 38876359.02556972, 'mld_mean': 35.1125372057195, 'mld_max': 123.23858642578125, 'mld_p90': 54.471997451782215}
(0, 43.0) {'duration': 6, 'intensity_mean': 1.2303421298958819, 'intensity_max': 3.2082361862371727, 'extent_mean': 5841072.131394535, 'extent_max': 10197047.295930523, 'extent_cumulative': 35046432.78836721, 'mld_mean': 40.123765836332545, 'mld_max': 158.23272705078125, 'mld_p90': 68.67738037109375}
(0, 58.0) {'duration': 20, 'intensity_mean': 1.1175425782794577, 'intensity_max': 3.25963174070813, 'extent_mean': 11007355.613692084, 'extent_max': 17351694.915759716, 'extent_cumulative': 220147112.27384168, 'mld_mean': 56.40374647746785, 'mld_max': 206.50076293945312, 'mld_p90': 106.9249496459961}
(0, 85.0) {'duration': 7, 'intensity_mean': 1.3049243401237254, 'intensity_max': 3.18635082809578

In [14]:
R = 6371.0
dlat_fosi = np.deg2rad(np.mean(np.diff(fosi_blobs.lat.values)))
dlon_fosi = np.deg2rad(np.mean(np.diff(fosi_blobs.lon.values)))
cell_area_fosi = (R**2) * dlat_fosi * dlon_fosi * np.cos(np.deg2rad(fosi_blobs.lat))
cell_area_da_fosi = xr.DataArray(cell_area_fosi.values, coords={'lat': fosi_blobs.lat}, dims='lat')

print(f"FOSI equatorial check: {float(cell_area_da_fosi.sel(lat=fosi_blobs.lat.values[len(fosi_blobs.lat)//2])):.1f} km²")

def compute_analog_characteristics(labels_da, analog_id, ssta_da, mld_da, cell_area_da):
    mask = (labels_da == analog_id)
    times_present = labels_da.time.where(mask.any(dim=('lat','lon')), drop=True)
    if len(times_present) == 0:
        return None

    mask_trimmed = mask.sel(time=times_present)
    ssta_masked = ssta_da.sel(time=times_present).where(mask_trimmed)
    extent_ts = (mask_trimmed.astype(int) * cell_area_da).sum(dim=('lat','lon'))

    return {
        'duration': len(times_present),
        'intensity_mean': float(ssta_masked.mean()),
        'intensity_max': float(ssta_masked.max()),
        'extent_mean': float(extent_ts.mean()),
        'extent_max': float(extent_ts.max()),
        'extent_cumulative': float(extent_ts.sum()),
    }

regridded_ssta_no_trend_NEP_da = regridded_ssta_no_trend_NEP['__xarray_dataarray_variable__']

blob_fosi_characteristics = compute_analog_characteristics(
    fosi_blobs, object_id, regridded_ssta_no_trend_NEP_da, None, cell_area_da_fosi
)
print(blob_fosi_characteristics)

FOSI equatorial check: 11881.4 km²
{'duration': 20, 'intensity_mean': 1.0101917806077365, 'intensity_max': 2.3448744454154564, 'extent_mean': 12854675.1150951, 'extent_max': 24261030.79022178, 'extent_cumulative': 257093502.301902}


In [45]:
blob_fosi_characteristics['mld_mean'] = 57.8
blob_fosi_characteristics['mld_max'] = 225.7
blob_fosi_characteristics['mld_p90'] = 103.8

print(blob_fosi_characteristics)

{'duration': 20, 'intensity_mean': 1.0101917806077365, 'intensity_max': 2.3448744454154564, 'extent_mean': 12854675.1150951, 'extent_max': 24261030.79022178, 'extent_cumulative': 257093502.301902, 'mld_mean': 57.8, 'mld_max': 225.7, 'mld_p90': 103.8}


In [47]:
values = np.array([c[char] for c in all_analog_characteristics.values() if char in c])